<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation, and observability trace extraction **without requiring any AWS access keys** from the recruiter. Credentials and secrets are retrieved securely via Cognito Identity Pools and AWS SSM.

In [41]:
# Install dependencies silently
!pip install boto3 requests -q
!sudo apt-get update -y -q > /dev/null && sudo apt-get install neofetch -y -q > /dev/null
!neofetch

import boto3, requests, json, urllib.parse, time

REGION = "us-east-1"
USER_POOL_ID = "us-east-1_5pCxpIkx8"
CLIENT_ID = "2r1ik1k110jbu6nfmuoegk5lns"
IDENTITY_POOL_ID = "us-east-1:c7680c24-fe96-4358-b305-6f43de1ca6c8"

access_token = None
id_token = None
username = ""
password = ""

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
            .-/+oossssoo+/-.
        `:+ssssssssssssssssss+:`
      -+ssssssssssssssssssyyssss+-
    .ossssssssssssssssssdMMMNysssso.
   /ssssssssssshdmmNNmmyNMMMMhssssss/
  +ssssssssshmydMMMMMMMNddddyssssssss+
 /sssssssshNMMMyhhyyyyhmNMMMNhssssssss/
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
 /sssssssshNMMMyhhyyyyhdNMMMNhssssssss/
  +sssssssssdmydMMMMMMMMddddyssssssss+
   /ssssssssssshdmNNNNmyNMMMMhssssss/
    .ossssssssssssssssssdMMMNysssso.
      -+sssssssssssssssssyyyssss+-
        `:+ssssssssssssssssss+:`
            .-/+oossssoo+/-.
root@d768b65d2734 
----------------- 
OS: Ubuntu 22.04.5 LTS x8

### 1. Retrieve Guest Login Credentials
We use the Cognito Identity Pool (Unauthenticated) to securely fetch the login credentials from AWS SSM.

In [ ]:
identity_client = boto3.client('cognito-identity', region_name=REGION)

try:
    # 1. Get Guest Identity ID
    id_resp = identity_client.get_id(IdentityPoolId=IDENTITY_POOL_ID)
    identity_id = id_resp['IdentityId']

    # 2. Get Guest Credentials (classic flow — bypasses Cognito session policy)
    sts = boto3.client('sts', region_name=REGION)
    cred_resp = sts.assume_role_with_web_identity(
        RoleArn='arn:aws:iam::162187491349:role/cognito_unauthenticated_role',
        RoleSessionName='GuestSession',
        WebIdentityToken=identity_id
    )
    guest_creds = cred_resp['Credentials']

    ssm_guest = boto3.client(
        'ssm', region_name=REGION,
        aws_access_key_id=guest_creds['AccessKeyId'],
        aws_secret_access_key=guest_creds['SecretAccessKey'],
        aws_session_token=guest_creds['SessionToken']
    )

    username = ssm_guest.get_parameter(Name='/financial-ai/analyst-username', WithDecryption=True)['Parameter']['Value']
    password = ssm_guest.get_parameter(Name='/financial-ai/analyst-password', WithDecryption=True)['Parameter']['Value']

    print("Success: retrieved demo credentials via Guest Identity.")
except Exception as e:
    print(f"Warning: Guest Retrieval failed ({str(e)[:120]}), using fallback.")
    username = "analyst_user"
    password = "FinAIAgent2026@"

### 2. Authenticate with Amazon Cognito
Login using the credentials retrieved in the previous step.

In [43]:
if not username or not password:
    print("Error: Credentials not found. Run Step 1 successfully first.")
else:
    client = boto3.client('cognito-idp', region_name=REGION)
    try:
        auth_response = client.initiate_auth(
            ClientId=CLIENT_ID,
            AuthFlow='USER_PASSWORD_AUTH',
            AuthParameters={'USERNAME': username, 'PASSWORD': password}
        )
        access_token = auth_response['AuthenticationResult']['AccessToken']
        id_token = auth_response['AuthenticationResult']['IdToken']
        print("Success: Cognito Authentication Successful.")
    except Exception as e:
        print(f"Error: Authentication Failed: {str(e)}")

Success: Cognito Authentication Successful.


### 3. Invoke the Agentcore Streaming Endpoint

In [ ]:
import uuid

AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-hvRgckAqaW"
encoded_arn = urllib.parse.quote(AGENT_ARN, safe='')
AGENTCORE_URL = f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{encoded_arn}/invocations"

# Master session ID for Langfuse trace grouping
MASTER_SESSION_ID = str(uuid.uuid4())
print(f"Session ID: {MASTER_SESSION_ID}")

def query_financial_agent(prompt: str):
    if access_token is None:
        print("Error: Access token is missing. Run Cell 2 successfully first.")
        return

    print(f"\n--- Query: {prompt} ---")
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": MASTER_SESSION_ID,
    }

    try:
        response = requests.post(AGENTCORE_URL, headers=headers, json={"prompt": prompt}, stream=True, timeout=120)
        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text[:300]}")
            return

        for line in response.iter_lines():
            if line:
                decoded = line.decode('utf-8')
                if decoded.startswith("data: "):
                    data = json.loads(decoded[6:])
                    if "event" in data:
                        print(data["event"])
                    elif "error" in data:
                        print(f"Agent error: {data['error'][:300]}")
    except Exception as e:
        print(f"An error occurred during invocation: {e}")

### 4. Execute Required Financial Queries

In [45]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]
for q in queries: query_financial_agent(q)


--- Query: What is the stock price for Amazon right now? ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: What were the stock prices for Amazon in Q4 last year? ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: Compare Amazon's recent stock performance to what analysts predicted in their reports. ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: I’m researching AMZN give me the current price and any relevant information about their AI business. ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: What is the total amount of office space Amazon owned in North America in 2024? ---
Error 424: {"message":"An error occurred when s

### 5. Observability Verification (Authenticated AWS Identity)
Fetch Langfuse traces securely using temporary credentials obtained via the Cognito session.

**Session ID:** `e9ab4997-f01b-41ca-abde-bfb7cf06c258`

In [ ]:
try:
    # 1. Exchange IdToken for Authenticated credentials
    id_resp = identity_client.get_id(
        IdentityPoolId=IDENTITY_POOL_ID,
        Logins={ f'cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}': id_token }
    )
    auth_creds = identity_client.get_credentials_for_identity(
        IdentityId=id_resp['IdentityId'],
        Logins={ f'cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}': id_token }
    )['Credentials']

    # 2. Use credentials to fetch Langfuse keys from SSM
    ssm_auth = boto3.client(
        'ssm', region_name=REGION,
        aws_access_key_id=auth_creds['AccessKeyId'],
        aws_secret_access_key=auth_creds['SecretKey'],
        aws_session_token=auth_creds['SessionToken']
    )

    pk = ssm_auth.get_parameter(Name='/financial-ai/langfuse/public-key', WithDecryption=True)['Parameter']['Value']
    sk = ssm_auth.get_parameter(Name='/financial-ai/langfuse/secret-key', WithDecryption=True)['Parameter']['Value']

    print(f"Success: retrieved Langfuse keys (PK: {pk[:7]}...)")

    if "placeholder" in sk or "00000000" in sk:
        print("NOTE: Langfuse keys are placeholders. Traces will not be available until real keys are stored in SSM.")
    else:
        print("Waiting 10s for trace propagation...")
        time.sleep(10)

        found = False
        for host in ["https://us.cloud.langfuse.com", "https://cloud.langfuse.com", "https://eu.cloud.langfuse.com"]:
            trace_url = f"{host}/api/public/sessions/{MASTER_SESSION_ID}"
            trace_resp = requests.get(trace_url, auth=(pk, sk))
            if trace_resp.status_code == 200:
                print(f"\n--- Langfuse Traces (Host: {host}, Session: {MASTER_SESSION_ID}) ---")
                print(json.dumps(trace_resp.json(), indent=2)[:2000])
                found = True
                break

        if not found:
            print(f"No traces found for session {MASTER_SESSION_ID}. Check Langfuse dashboard manually.")
except Exception as e:
    print(f"Error: Observability Retrieval Failed: {str(e)}")